# 04 — Features de morphologie urbaine

Reproduit la section **Urban morphology metrics and density surfaces** du papier,
avec leurs paramètres exacts :

| Métrique du papier | Paramètre | Notre implémentation |
|---|---|---|
| Continuous building density | R = 300 m, normalisé par πR² (bâtiments/km²) | OSMnx buildings + buffer 300 m |
| Road density | km de route / km² dans le rayon R | OSMnx edges + buffer 300 m |
| Intersection count | nb d'intersections dans le rayon R | OSMnx nodes + buffer 300 m |
| Sample-level extraction | buffer 100 m autour de chaque point | buffer 100 m pour le near-field |
| Slope (DEM) | pente locale moyenne | omis (Kampala/Hanoï : impact faible, à noter comme limitation) |

Ces features sont **aussi** les inputs du surrogate model (notebook 06) — c'est la même chose.

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import os, warnings
warnings.filterwarnings('ignore')

R = 300            # rayon du papier (m)
NEAR_FIELD = 100   # buffer sample-level du papier (m)
AREA_KM2 = np.pi * (R / 1000) ** 2   # aire du disque en km²

# CRS projetés en mètres : Kampala = UTM 36N, Hanoï = UTM 48N
CRS_KAMPALA = 'EPSG:32636'
CRS_HANOI   = 'EPSG:32648'

# Données nettoyées du notebook 02
df = pd.read_csv('../data/processed/uganda/sunbird_clean.csv')
print(f'{len(df)} échantillons')

## Téléchargement OSM (une seule fois, mis en cache)

On limite au rectangle couvrant les points + marge de 500 m, sinon Kampala entier est trop lourd.

In [ ]:
# Téléchargement par région (Kampala et Entebbe sont à ~20 km — une seule bbox
# engloberait le lac Victoria et des zones inutiles)
MARGIN = 0.01  # ~1 km

osm = {}  # region -> (buildings, edges, nodes)
for region, g in df.groupby('region'):
    bbox = (g.longitude.min() - MARGIN, g.latitude.min() - MARGIN,
            g.longitude.max() + MARGIN, g.latitude.max() + MARGIN)

    bpath = f'../data/processed/{region.lower()}_buildings.gpkg'
    gpath = f'../data/processed/{region.lower()}_roads.graphml'

    if not os.path.exists(bpath):
        print(f'{region} : téléchargement bâtiments OSM (plusieurs minutes la 1re fois)...')
        b = ox.features_from_bbox(bbox, tags={'building': True})
        b = b[b.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
        b[['geometry']].to_file(bpath, driver='GPKG')
    buildings = gpd.read_file(bpath)

    if not os.path.exists(gpath):
        print(f'{region} : téléchargement réseau routier...')
        G = ox.graph_from_bbox(bbox, network_type='drive')
        ox.save_graphml(G, gpath)
    G = ox.load_graphml(gpath)
    nodes, edges = ox.graph_to_gdfs(G)

    osm[region] = (buildings, edges, nodes)
    print(f'{region} : {len(buildings)} bâtiments, {len(edges)} segments, {len(nodes)} intersections')

## Calcul vectorisé des métriques du papier

On projette tout en UTM (mètres), on crée un buffer de 300 m autour de chaque point,
et on compte avec un spatial join (rapide même avec des milliers de points).

In [ ]:
def morphology_features(df_points, buildings, edges, nodes, crs_utm):
    """Calcule les métriques du papier pour chaque point GPS (index remis à zéro)."""
    pts = gpd.GeoDataFrame(
        df_points.reset_index(drop=True).copy(),
        geometry=gpd.points_from_xy(df_points.longitude, df_points.latitude),
        crs='EPSG:4326'
    ).to_crs(crs_utm)

    bld = buildings.to_crs(crs_utm)
    bld = bld.set_geometry(bld.geometry.centroid)
    edg = edges.to_crs(crs_utm).reset_index(drop=True)
    nod = nodes.to_crs(crs_utm).reset_index(drop=True)

    # Buffer R=300m autour de chaque point
    buf = gpd.GeoDataFrame(
        {'pt_id': range(len(pts))},
        geometry=pts.geometry.buffer(R), crs=crs_utm)

    # 1. Building density : bâtiments dans R / (pi*R^2) — métrique exacte du papier
    jb = gpd.sjoin(bld[['geometry']], buf, predicate='within').groupby('pt_id').size()
    pts['building_density_km2'] = [jb.get(i, 0) / AREA_KM2 for i in range(len(pts))]

    # 2. Road density : km de route intersectant le buffer / km^2
    #    (longueur totale des segments touchant le disque — légère surestimation
    #     vs découpe exacte au cercle, acceptable et noté comme telle)
    jr = gpd.sjoin(edg[['geometry']], buf, predicate='intersects')
    rl = jr.groupby('pt_id').apply(lambda g: g.geometry.length.sum())
    pts['road_density_km_km2'] = [(rl.get(i, 0) / 1000) / AREA_KM2 for i in range(len(pts))]

    # 3. Intersection count dans R
    jn = gpd.sjoin(nod[['geometry']], buf, predicate='within').groupby('pt_id').size()
    pts['intersection_count'] = [jn.get(i, 0) for i in range(len(pts))]

    # 4. Distance à la route la plus proche (near-field du papier)
    pts['dist_road_m'] = pts.geometry.apply(lambda p: edg.distance(p).min())

    return pd.DataFrame(pts.drop(columns='geometry'))

# Calcul par région puis concaténation (UTM 36N valable pour les deux villes)
feats = []
for region, (buildings, edges, nodes) in osm.items():
    sub = df[df['region'] == region]
    print(f'{region} : calcul sur {len(sub)} points...')
    feats.append(morphology_features(sub, buildings, edges, nodes, CRS_KAMPALA))

feat = pd.concat(feats, ignore_index=True)
feat.to_parquet('../data/processed/uganda/sunbird_morphology.parquet', index=False)
print('Features sauvegardées.')
feat[['noise_measurement', 'building_density_km2', 'road_density_km_km2',
      'intersection_count', 'dist_road_m']].describe().round(1)

## Vérification : morphologie vs bruit (comme la fin du papier)

Le papier conclut : densité bâtie et intensité du réseau routier sont **positivement corrélées au SPL**.
On vérifie qu'on retrouve la même chose.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols = ['building_density_km2', 'road_density_km_km2', 'intersection_count', 'dist_road_m']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, cols):
    ax.scatter(feat[c], feat['noise_measurement'], alpha=0.2, s=8)
    r = feat[[c, 'noise_measurement']].corr().iloc[0, 1]
    ax.set_title(f'{c}\nr = {r:.2f}')
    ax.set_ylabel('dB')
plt.tight_layout()
plt.savefig('../results/figures/sunbird/morphology_vs_spl.png', dpi=150)
plt.show()

**Attendu (d'après le papier)** : corrélation positive pour building density, road density,
intersections — négative pour dist_road. Si on retrouve ça, la reproduction est validée
et ces colonnes deviennent les features du surrogate model (notebook 06).

**Limitation à noter** : le papier utilise aussi la pente (DEM) — omise ici, Hanoï est plate.